In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

In [ ]:
data = pd.read_csv("all_participants_data.csv", index_col="Participant")
filteredData = data.drop(columns=['median_arousal', 'median_valence'])
targetData = data['median_arousal']

indexes = data.index

scaler = StandardScaler()
scaledArray = scaler.fit_transform(filteredData)
scaledData = pd.DataFrame(scaledArray, index=indexes)

print(scaledData)

## Linear Regression Model

In [ ]:
#Linear Regression Model

from sklearn.linear_model import LinearRegression

model1 = LinearRegression()


## Neural Network Regressor

In [ ]:
#Neural Network Regressor

from sklearn.neural_network import MLPRegressor

model2 = MLPRegressor(
    hidden_layer_sizes=(16, 8),   # 2 hidden layers
    activation='relu',                 # activation function
    solver='adam',                     # optimizer
    max_iter=100,                      # max training iterations
    random_state=42
)

## LSTM Regressor

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

class LSTMRegressor(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,        # use the passed value
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True
        )
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # x: (batch, seq_len, input_size)
        lstm_out, _ = self.lstm(x)
        last = lstm_out[:, -1, :]
        return self.fc(last)
    

model3 = LSTMRegressor(1, 64, 3, 1)


In [ ]:
indexes = data.index
participants = set(indexes)

print(participants)

dataSubsets = [data.loc[participant] for participant in participants]


In [ ]:
from scipy.stats import pearsonr
import numpy as np

def CCcoefficient(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    mean_true = np.mean(y_true)
    mean_pred = np.mean(y_pred)
    var_true = np.var(y_true)
    var_pred = np.var(y_pred)
    cov = np.mean((y_true - mean_true) * (y_pred - mean_pred))

    ccc = (2 * cov) / (var_true + var_pred + (mean_true - mean_pred) ** 2)
    return ccc

# **Model Tests**

In [ ]:
# testing model 1

participants = list(set(indexes))

results = []

X = scaledData
Y = targetData

for participant in participants:
    trainIdx = [p for p in participants if p != participant]
    testIdx = participant

    Xtrain, Ytrain = X.loc[trainIdx], Y.loc[trainIdx]
    Xtest, Ytest = X.loc[testIdx], Y.loc[testIdx]

    model1.fit(Xtrain, Ytrain)

    Ypred = model1.predict(Xtest)

    results.append((pearsonr(np.array(Ytest).ravel(), np.array(Ypred).ravel())[0], CCcoefficient(Ytest, Ypred)))

print(results)
    

In [ ]:
# testing model 2

participants = list(set(indexes))

results = []

X = scaledData
Y = targetData

for participant in participants:
    trainIdx = [p for p in participants if p != participant]
    testIdx = participant

    Xtrain, Ytrain = X.loc[trainIdx], Y.loc[trainIdx]
    Xtest, Ytest = X.loc[testIdx], Y.loc[testIdx]

    model2.fit(Xtrain, Ytrain)

    Ypred = model2.predict(Xtest)

    results.append((pearsonr(np.array(Ytest).ravel(), np.array(Ypred).ravel())[0], CCcoefficient(Ytest, Ypred)))

print(results)

In [ ]:
# testing model 3

from torch.utils.data import TensorDataset, DataLoader

X = scaledData.to_numpy()
Y = targetData.to_numpy()

participants = np.array(list(set(indexes)))
n_features = scaledData.shape[1]           # number of columns in X

X_tensor = torch.tensor(X, dtype=torch.float32)
y_tensor = torch.tensor(Y, dtype=torch.float32)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

results = []
for participant in participants:
    test_mask = indexes == participant
    train_mask = ~test_mask

    Xtrain = X_tensor[train_mask].view(-1, 1, n_features).to(device)
    Ytrain = y_tensor[train_mask].view(-1, 1).to(device)
    Xtest  = X_tensor[test_mask].view(-1, 1, n_features).to(device)
    Ytest  = y_tensor[test_mask].view(-1, 1).to(device)

    model = LSTMRegressor(
        input_size=n_features,
        hidden_size=16,
        num_layers=2,
        output_size=1
    ).to(device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3)

    train_loader = DataLoader(
        TensorDataset(Xtrain, Ytrain),
        batch_size=64,      
        shuffle=True
    )

    for epoch in range(10):
        model.train()
        for xb, yb in train_loader:
            optimizer.zero_grad()
            y_pred = model(xb)
            loss = criterion(y_pred, yb)
            loss.backward()
            optimizer.step()

    model.eval()
    with torch.no_grad():
        Ypred_test = model(Xtest).cpu().numpy().ravel()
        Ytrue      = Ytest.cpu().numpy().ravel()

    results.append((
        pearsonr(Ytrue, Ypred_test)[0],
        CCcoefficient(Ytrue, Ypred_test)
    ))

print(results)
        

